# 비공개 r(평균 제구 성공률) 역추산

리더보드 점수 산식:

```
Score = max(0, 100000 × (1 - Brier / (r(1-r))))
Brier = mean((p_i - y_i)^2)
```

`r`은 비공개다. 정리하면:

```
Brier / (r(1-r)) = 1 - Score/100000
r(1-r) = Brier / (1 - Score/100000)          # =: K
r^2 - r + K = 0
r = (1 ± sqrt(1 - 4K)) / 2
```

**주의**: 상수 제출(전부 0, 전부 0.5, 전부 1)로는 절대 역추산이 안 된다. 상수 c에 대해

```
Brier(c) - r(1-r) = (c - r)^2 ≥ 0
```

이라서 상수 예측의 Brier는 항상 베이스라인 이상이고 Score는 항상 0 이하 → `max(0, ...)`에 걸려 전부 "0"으로만 보인다 (세 버전이 전부 똑같이 0으로 나와서 구분 불가). 아래 셀에서 직접 확인.

역산이 되려면 **실제로 베이스라인보다 나은(양수 점수) 모델을 제출**해야 하고, 그 모델의 로컬 Brier(OOT 검증값)를 알고 있어야 한다. 그 값과 실제 리더보드 점수를 아래 함수에 넣으면 r을 추정할 수 있다.

In [1]:
def brier_constant(c: float, r: float) -> float:
    """상수 예측 c의 이론적 Brier (참값 y_i는 베르누이(r)라고 가정)."""
    return r * (1 - 2 * c) + c ** 2


def score_formula(brier: float, r: float) -> float:
    baseline = r * (1 - r)
    return 100000 * (1 - brier / baseline)


# r을 안다고 가정하고(실제로는 비공개) 상수 제출이 전부 0으로 깎이는지 확인
assumed_r = 0.48
for c in [0.0, 0.5, 1.0]:
    raw = score_formula(brier_constant(c, assumed_r), assumed_r)
    print(f"c={c:>3}  raw_score={raw:>10.1f}  floored={max(0, raw):.1f}")

c=0.0  raw_score=  -92307.7  floored=0.0
c=0.5  raw_score=    -160.3  floored=0.0
c=1.0  raw_score= -108333.3  floored=0.0


위 결과에서 세 버전 모두 `raw_score`는 음수(또는 0)이고, `floored`는 전부 0으로 동일하게 나온다. 즉 상수 제출 3개를 실제로 리더보드에 올려봐도 셋 다 "0점"만 보이므로 그 안에서 `r`을 구분해낼 정보가 없다.

## r 사전 추정치 (train 추세 외삽)

역산 공식은 두 개의 근(r, 1-r)이 대칭이라 어느 쪽이 맞는지 고를 기준이 필요하다. `train.csv`의 연도별 성공률 추세를 선형 외삽해서 2025 사전 추정치로 쓴다.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().resolve().parent / "data"
train_df = pd.read_csv(DATA_DIR / "train.csv", usecols=["season", "control_success"])

season_rate = train_df.groupby("season")["control_success"].mean()
print(season_rate)

coef = np.polyfit(season_rate.index, season_rate.values, deg=1)
prior_r_2025 = np.polyval(coef, 2025)
print(f"\n연도당 변화율: {coef[0]:+.5f}/year")
print(f"2025 선형 외삽 사전추정치 prior_r = {prior_r_2025:.4f}")

season
2019    0.564670
2020    0.532712
2021    0.532762
2022    0.528920
2023    0.499957
2024    0.486105
Name: control_success, dtype: float64

연도당 변화율: -0.01414/year
2025 선형 외삽 사전추정치 prior_r = 0.4747


## r 역산 함수

In [3]:
def solve_r(brier_local: float, score: float, prior_r: float):
    """로컬 Brier와 실제 리더보드 점수로부터 r을 역산.
    두 근(r, 1-r) 중 prior_r에 더 가까운 쪽을 채택.
    """
    if score >= 100000:
        raise ValueError("score가 100000 이상이면 Brier=0(완벽 예측)이라 공식이 정의 안 됨")
    K = brier_local / (1 - score / 100000)
    disc = 1 - 4 * K
    if disc < 0:
        raise ValueError(
            f"K={K:.6f} > 0.25 → 실수해 없음. "
            "local Brier가 실제 public Brier와 많이 다르거나 입력값을 잘못 넣었을 가능성."
        )
    root_lo = (1 - disc ** 0.5) / 2
    root_hi = (1 + disc ** 0.5) / 2
    chosen = root_lo if abs(root_lo - prior_r) < abs(root_hi - prior_r) else root_hi
    return {"root_lo": root_lo, "root_hi": root_hi, "chosen": chosen, "K": K}


def triangulate_r(submissions: list[tuple[float, float]], prior_r: float):
    """여러 제출의 (local_brier, score) 쌍으로 r을 각각 추정하고 편차를 확인.
    추정치들이 서로 많이 벌어지면 local Brier가 public Brier의 신뢰할 만한 프록시가 아니라는 신호.
    """
    estimates = [solve_r(b, s, prior_r)["chosen"] for b, s in submissions]
    return {
        "estimates": estimates,
        "mean": float(np.mean(estimates)),
        "std": float(np.std(estimates)) if len(estimates) > 1 else 0.0,
    }

## 사용법

실제로 모델을 제출해서 리더보드 점수를 받으면 아래 셀의 `brier_local`, `score`를 채워서 실행.
`brier_local`은 우리가 로컬 OOT(2024) 검증에서 측정한 Brier 값 — public 테스트셋 Brier와 완전히 같진 않으니 근사치로 이해할 것.

In [4]:
# 예시 (실제 제출 후 숫자로 교체)
brier_local = 0.2480   # 로컬 OOT Brier
score = 700.0           # 실제 리더보드 점수

result = solve_r(brier_local, score, prior_r=prior_r_2025)
print(result)
print(f"\n추정 r ≈ {result['chosen']:.4f} (train 추세 외삽 prior={prior_r_2025:.4f})")

{'root_lo': 0.48413297960061535, 'root_hi': 0.5158670203993847, 'chosen': 0.48413297960061535, 'K': 0.2497482376636455}

추정 r ≈ 0.4841 (train 추세 외삽 prior=0.4747)


In [5]:
# 제출을 여러 번 했다면 (local_brier, score) 쌍을 리스트로 넣어서 일관성 확인
# submissions = [(0.2480, 700.0), (0.2479, 715.0), (0.2478, 730.0)]
# triangulate_r(submissions, prior_r=prior_r_2025)